In [ ]:
# input
aligned_result = "../../data/pred_cluster_aligned_result.tsv"
rep_info_file = "../../data/repId-numPro-numPredPro-numResi-numPredResi-numAnnoPro-isAnno.tsv"

# output
rep_clique_file = "./tmp/repId-cliques.tsv"

In [2]:
import pandas as pd

# resi / pro == pred_resi / pred_pro
df_rep_info = pd.read_table(rep_info_file, header=None, names=["rep_id", "num_pro", "num_pred_pro", "num_resi", "num_pred_resi", "num_anno_pro", "is_anno"])
perfect_aligned_pred_rep_ids = df_rep_info[
    df_rep_info.apply(
        lambda row: (row['num_resi'] / row['num_pro']) == (row['num_pred_resi'] / row['num_pred_pro']), 
        axis=1
    )
]['rep_id']
perfect_aligned_pred_rep_ids = set(perfect_aligned_pred_rep_ids)
len(perfect_aligned_pred_rep_ids)

df_aligned = pd.read_table(aligned_result)
df_aligned = df_aligned[df_aligned['rep_id'].map(lambda x: x not in perfect_aligned_pred_rep_ids)]

len(df_aligned)


6643353

17284249

In [3]:
cur_reps = set(df_aligned['rep_id'])
len(cur_reps)
df_rep_info = df_rep_info[df_rep_info['rep_id'].map(lambda x: x in cur_reps)]

# only consider seq cluster size no bigger than 1000
# for C4P9C8 (23759), it took ~5 days in clique search
df_rep_info = df_rep_info[df_rep_info['num_pro'] <= 1000]
cur_reps = set(df_rep_info['rep_id'])
len(cur_reps)

df_aligned = df_aligned[df_aligned['rep_id'].map(lambda x: x in cur_reps)]
len(df_aligned)

df_pred = df_aligned[df_aligned['is_pred'].map(lambda x: x.count("1") >= 3)]
len(df_pred)

772481

771093

14389748

12823221

In [4]:
import gc

del df_rep_info
del perfect_aligned_pred_rep_ids
del df_aligned
del cur_reps
gc.collect()

0

## conservation and divergence in predicted proteins

In [ ]:
from itertools import combinations
import networkx as nx
import tqdm
import numpy as np

def split_dataframe_balanced(df, column='num_pro', n=3):
    groups = [[] for _ in range(n)]
    group_sums = [0 for _ in range(n)]

    sorted_df = df.sort_values(by=column, ascending=False)

    for idx, row in sorted_df.iterrows():
        min_index = np.argmin(group_sums)
        groups[min_index].append(idx)
        group_sums[min_index] += row[column]

    dfs = [df.loc[grp].reset_index(drop=True) for grp in groups]
    return dfs, group_sums


# some flaws:
# ooooooooo
# ---oooooo
# ooo---ooo

def find_pred_cliques(
    df: pd.DataFrame, 
    file: str,
):
    records = []
    for (rep_id,), df_rep in tqdm.tqdm(df.groupby(by=["rep_id"])):

        pred_idxes = [{idx for idx, value in enumerate(x.split(",")) if value == "1"} for x in df_rep["is_pred"]]

        if len(pred_idxes) == 1:
            records.append({
                "rep_id": rep_id,
                "cliques": rep_id
            })
            continue

        g = nx.Graph()
        for i, j in combinations(range(len(pred_idxes)), 2):
            id_i = df_rep['seq_id'].iloc[i]
            id_j = df_rep['seq_id'].iloc[j]

            pred_idx_i = pred_idxes[i]
            pred_idx_j = pred_idxes[j]
            if len(pred_idx_i & pred_idx_j) >= 3:
                g.add_edge(id_i, id_j)
            else:
                g.add_node(id_i)
                g.add_node(id_j)

        cliques = list(nx.algorithms.find_cliques(g))
        records.append({
            "rep_id": rep_id,
            "cliques": ";".join([",".join(c) for c in cliques])
        })

    pd.DataFrame(records).to_csv(file, sep="\t", index=None)

In [6]:
import multiprocessing as mp
import numpy as np

num_task = 12
chunks = np.array_split(list(set(df_pred['rep_id'])), num_task)
dfs = [df_pred[df_pred['rep_id'].isin(chunk)] for chunk in chunks]
jobs = mp.Pool(num_task)
result = [jobs.apply_async(find_pred_cliques, args=(df, f"{rep_clique_file}.part_{idx}")) for idx, df in enumerate(dfs)]
result = [r.get() for r in result]

100%|██████████| 64258/64258 [39:23<00:00, 27.19it/s] 
